Deduplication (Day 9).
Reads:  jobmarket.silver.stg_postings_api, stg_postings_kaggle
Writes: same tables, deduped, with posting_id
Kind 1: mechanical (same source_posting_id, re-ingestion) -> keep latest
Kind 2: semantic (same company+title_norm+city+week) -> keep most complete

In [0]:
from pyspark.sql import functions as F

api = spark.table("jobmarket.silver.stg_postings_api")

total = api.count()
distinct_ids = api.select("source_posting_id").distinct().count()

print(f"Rows: {total}")
print(f"Distinct source IDs: {distinct_ids}")
print(f"Mechanical duplicates: {total - distinct_ids} "
      f"({(total - distinct_ids)/total*100:.1f}%)")

In [0]:
(api.groupBy("source_posting_id").count()
    .filter("count > 1")
    .join(api, "source_posting_id")
    .select("source_posting_id", "title_raw", "company", "ingest_ts")
    .display())

In [0]:
from pyspark.sql.window import Window

# Window spec: group rows by source_posting_id, order within each group
# by ingest_ts descending (newest first)
w = Window.partitionBy("source_posting_id").orderBy(F.col("ingest_ts").desc())

api_deduped = (api
    .withColumn("rn", F.row_number().over(w))   # 1 = newest in its group
    .filter(F.col("rn") == 1)
    .drop("rn"))

print(f"Before: {api.count()}  After: {api_deduped.count()}")

In [0]:
# ---- Semantic dedup key: company + title_norm + city + week ----
# Null company or city -> EXEMPT from semantic dedup (distinct anonymous
# posters must not collapse into one job; conservative > destructive)

api_keyed = (api_deduped
    .withColumn("posted_week", F.date_trunc("week", F.col("posted_date")))
    .withColumn("dedup_eligible",
        F.col("company").isNotNull() & F.col("city").isNotNull())
    .withColumn("semantic_key",
        F.when(F.col("dedup_eligible"),
            F.concat_ws("||",
                F.lower(F.trim("company")),
                F.col("title_norm"),
                F.coalesce(F.col("seniority"), F.lit("none")),
                F.lower(F.trim("city")),
                F.col("posted_week").cast("string")))
         .otherwise(F.concat_ws("||", F.lit("nokey"), "source_posting_id"))))

# quick shape check before measuring anything
api_keyed.select("company", "title_norm", "city", "posted_week",
                 "dedup_eligible", "semantic_key").limit(5).display()

In [0]:
eligible = api_keyed.filter("dedup_eligible")

n_eligible  = eligible.count()
n_jobs      = eligible.select("semantic_key").distinct().count()
n_dupes     = n_eligible - n_jobs

print(f"Eligible rows:        {n_eligible}")
print(f"Distinct jobs (keys): {n_jobs}")
print(f"Semantic duplicates:  {n_dupes} ({n_dupes/n_eligible*100:.1f}%)")

# spot-check: the biggest duplicate groups — are they REALLY the same job?
(api_keyed.groupBy("semantic_key").count()
    .filter("count > 1")
    .orderBy(F.desc("count"))
    .limit(5)
    .join(api_keyed, "semantic_key")
    .select("semantic_key", "title_raw", "company", "city",
            "posted_date", "salary_min", "salary_max")
    .orderBy("semantic_key", "posted_date")
    .display())

In [0]:
# completeness score: 2 = both salary bounds, 1 = one, 0 = none
completeness = (F.col("salary_min").isNotNull().cast("int") +
                F.col("salary_max").isNotNull().cast("int"))

w_sem = (Window.partitionBy("semantic_key")
               .orderBy(F.desc(completeness), F.col("ingest_ts").desc()))

api_final = (api_keyed
    .withColumn("rn", F.row_number().over(w_sem))
    .filter("rn = 1")
    .drop("rn", "dedup_eligible")
    # deterministic posting_id: same key -> same hash, every run (MERGE fuel)
    .withColumn("posting_id", F.sha2(F.col("semantic_key"), 256)))

print(f"Before: {api_keyed.count()}  After: {api_final.count()}")
api_final.select("posting_id", "semantic_key", "salary_min", "salary_max").limit(3).display()

In [0]:
kaggle = spark.table("jobmarket.silver.stg_postings_kaggle")

# same key construction — one difference: Kaggle has no mechanical-dupe
# concern (single load), so we go straight to semantic
kaggle_keyed = (kaggle
    .withColumn("posted_week", F.date_trunc("week", F.col("posted_date")))
    .withColumn("dedup_eligible",
        F.col("company").isNotNull() & F.col("city").isNotNull())
    .withColumn("semantic_key",
        F.when(F.col("dedup_eligible"),
            F.concat_ws("||",
                F.lower(F.trim("company")),
                F.col("title_norm"),
                F.coalesce(F.col("seniority"), F.lit("none")),
                F.lower(F.trim("city")),
                F.col("posted_week").cast("string")))
         .otherwise(F.concat_ws("||", F.lit("nokey"), "source_posting_id"))))

eligible_k = kaggle_keyed.filter("dedup_eligible")
n_elig = eligible_k.count()
n_jobs = eligible_k.select("semantic_key").distinct().count()
print(f"Eligible: {n_elig}   Distinct jobs: {n_jobs}")
print(f"Semantic duplicates: {n_elig - n_jobs} ({(n_elig - n_jobs)/n_elig*100:.1f}%)")

# spot-check, same discipline
(kaggle_keyed.groupBy("semantic_key").count()
    .filter("count > 1").orderBy(F.desc("count")).limit(5)
    .join(kaggle_keyed, "semantic_key")
    .select("semantic_key", "title_raw", "posted_date", "salary_min", "salary_max")
    .orderBy("semantic_key", "posted_date")
    .display())

In [0]:
completeness_k = (F.col("salary_min").isNotNull().cast("int") +
                  F.col("salary_max").isNotNull().cast("int"))

w_k = (Window.partitionBy("semantic_key")
             .orderBy(F.desc(completeness_k), F.col("ingest_ts").desc()))

kaggle_final = (kaggle_keyed
    .withColumn("rn", F.row_number().over(w_k))
    .filter("rn = 1")
    .drop("rn", "dedup_eligible")
    .withColumn("posting_id", F.sha2(F.col("semantic_key"), 256)))

print(f"Kaggle before: {kaggle_keyed.count()}  after: {kaggle_final.count()}")

# write both deduped tables back to staging
(api_final.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("jobmarket.silver.stg_postings_api"))

(kaggle_final.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("jobmarket.silver.stg_postings_kaggle"))

print("Both staging tables deduped and written.")